<a href="https://colab.research.google.com/github/shwetakul2005/mental-health-project/blob/main/01_video_file_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tarfile
import urllib.request
import io

base_url = "https://dcapswoz.ict.usc.edu/wwwedaic/data/"
session_id = "300"

# Stream the tar.gz without saving it fully
url = f"{base_url}{session_id}_P.tar.gz"
response = urllib.request.urlopen(url)
buffer = io.BytesIO(response.read())

In [ ]:
# with tarfile.open(fileobj=buffer) as tar:
#     # See what's inside first
#     print(tar.getnames())

#     # Extract only the transcript file
#     transcript = tar.extractfile(f"{session_id}_P/{session_id}_Transcript.csv")
#     content = transcript.read().decode("utf-8")

['300_P', '300_P/300_AUDIO.wav', '300_P/features', '300_P/features/300_CNN_VGG.mat', '300_P/features/300_densenet201.csv', '300_P/features/300_OpenSMILE2.3.0_mfcc.csv', '300_P/features/300_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv', '300_P/features/300_vgg16.csv', '300_P/features/300_BoAW_openSMILE_2.3.0_eGeMAPS.csv', '300_P/features/300_BoAW_openSMILE_2.3.0_MFCC.csv', '300_P/features/300_OpenFace2.1.0_Pose_gaze_AUs.csv', '300_P/features/300_OpenSMILE2.3.0_egemaps.csv', '300_P/features/300_CNN_ResNet.mat', '300_P/300_Transcript.csv']


In [ ]:
import requests
from bs4 import BeautifulSoup

labels_url = "https://dcapswoz.ict.usc.edu/wwwedaic/labels/"
response = requests.get(labels_url)
soup = BeautifulSoup(response.text, "html.parser")

# List all files
for link in soup.find_all("a"):
    href = link.get("href")
    if href and href != "../":
        print(href)

?C=N;O=D
?C=M;O=A
?C=S;O=A
?C=D;O=A
/wwwedaic/
Detailed_PHQ8_Labels.csv
detailed_lables.csv
dev_split.csv
test_split.csv
train_split.csv


In [ ]:
import pandas as pd
import io

def peek_csv(filename):
    url = f"https://dcapswoz.ict.usc.edu/wwwedaic/labels/{filename}"
    r = requests.get(url)
    df = pd.read_csv(io.StringIO(r.text))
    print(f"\n=== {filename} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(3))
    return df

# Also peek metadata
meta = peek_csv("../metadata_mapped.csv")


=== ../metadata_mapped.csv ===
Shape: (219, 7)
Columns: ['Participant_ID', 'AVECParticipant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']
   Participant_ID AVECParticipant_ID  Gender  PHQ_Binary  PHQ_Score  \
0             302       training_001    male           0          4   
1             303       training_002  female           0          0   
2             304       training_003  female           0          6   

   PCL-C (PTSD)  PTSD Severity  
0             0             28  
1             0             17  
2             0             20  


In [ ]:
import tarfile

r = requests.get("https://dcapswoz.ict.usc.edu/wwwedaic/labels2019.tar.gz")
buffer = io.BytesIO(r.content)  # only 3.2KB so fine

with tarfile.open(fileobj=buffer) as tar:
    print("Files inside labels2019.tar.gz:")
    for name in tar.getnames():
        print(" ", name)

    # Read each file inside
    for member in tar.getmembers():
        if member.name.endswith(".csv"):
            f = tar.extractfile(member)
            df = pd.read_csv(io.StringIO(f.read().decode("utf-8")))
            print(f"\n=== {member.name} ===")
            print(f"Shape: {df.shape}")
            print(f"Columns: {list(df.columns)}")
            print(df.head(3))

Files inside labels2019.tar.gz:
  labels
  labels/Detailed_PHQ8_Labels.csv
  labels/test_split.csv
  labels/dev_split.csv
  labels/train_split.csv

=== labels/Detailed_PHQ8_Labels.csv ===
Shape: (219, 10)
Columns: ['Participant_ID', 'PHQ_8NoInterest', 'PHQ_8Depressed', 'PHQ_8Sleep', 'PHQ_8Tired', 'PHQ_8Appetite', 'PHQ_8Failure', 'PHQ_8Concentrating', 'PHQ_8Moving', 'PHQ_8Total']
   Participant_ID  PHQ_8NoInterest  PHQ_8Depressed  PHQ_8Sleep  PHQ_8Tired  \
0             300                0               0           1           0   
1             301                0               0           1           1   
2             302                1               1           0           1   

   PHQ_8Appetite  PHQ_8Failure  PHQ_8Concentrating  PHQ_8Moving  PHQ_8Total  
0              1             0                   0            0           2  
1              1             0                   0            0           3  
2              0             1                   0            0        

In [ ]:
# %%bash
# # This only downloads enough bytes to read the tar headers — stops early
# curl -s "https://dcapswoz.ict.usc.edu/wwwedaic/data/300_P.tar.gz"
#   | tar -tzf - 2>/dev/null | head -30

In [ ]:
FILES_TO_EXTRACT = [
    "{sid}_Transcript.csv",                              # NLP
    "features/{sid}_OpenFace2.1.0_Pose_gaze_AUs.csv",  # Video frame-level
    "features/{sid}_OpenSMILE2.3.0_egemaps.csv",        # Audio (eGeMAPS better than MFCC for affect)
    "features/{sid}_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv",  # Video aggregated
    "features/{sid}_BoAW_openSMILE_2.3.0_eGeMAPS.csv", # Audio aggregated
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import requests
import tarfile
import io
import os
import pandas as pd

BASE_URL = "https://dcapswoz.ict.usc.edu/wwwedaic/data/"
SAVE_DIR = "/content/drive/MyDrive/edaic"  # change to your path
os.makedirs(SAVE_DIR, exist_ok=True)


def download_session(session_id):
    sid = str(session_id)
    url = f"{BASE_URL}{sid}_P.tar.gz"
    out_dir = os.path.join(SAVE_DIR, sid)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n[{sid}] Downloading...", end=" ")
    response = requests.get(url, stream=True)
    if response.status_code != 200:
        print(f"FAILED ({response.status_code})")
        return

    # Buffer the full tar
    chunks = []
    for chunk in response.iter_content(chunk_size=1024*1024):
        chunks.append(chunk)
    buffer = io.BytesIO(b"".join(chunks))
    print("Done. Extracting...")

    with tarfile.open(fileobj=buffer) as tar:
        for file_template in FILES_TO_EXTRACT:
            fname = file_template.format(sid=sid)
            tar_path = f"{sid}_P/{fname}"
            try:
                member = tar.getmember(tar_path)
                f = tar.extractfile(member)
                if f:
                    # Flatten into out_dir, preserve features/ subdir
                    out_path = os.path.join(out_dir, fname)
                    os.makedirs(os.path.dirname(out_path), exist_ok=True)
                    with open(out_path, "wb") as out_file:
                        out_file.write(f.read())
                    print(f"  ✓ {fname}")
            except KeyError:
                print(f"  ✗ {fname} not found")



In [ ]:
# Run for all sessions — load IDs from your labels CSV
# The train_split.csv is already loaded into the 'df' variable from a previous cell.
labels = df # Use the DataFrame 'df' which contains the train_split.csv data
session_ids = labels["Participant_ID"].tolist()

for sid in session_ids:
    download_session(sid)

In [ ]:
"""
E-DAIC Video Modality Feature Extraction Pipeline
===================================================
Source file : XXX_OpenFace2.1.0_Pose_gaze_AUs.csv  (one per session)
Output      : session-level feature vectors ready for fusion model

Feature groups extracted
------------------------
1. Action Units (AU)     — 17 AUs, continuous intensity (r) + binary presence (c)
2. Head pose             — Rx, Ry, Rz  (pitch, yaw, roll in radians)
3. Gaze                  — gaze_angle_x, gaze_angle_y
4. Derived / dynamic     — smile composite, negative affect composite,
                           AU activation rates, head movement velocity,
                           gaze instability score

Each group is aggregated with: mean, std, min, max, range, median
Final output per session : ~220-dim vector  (exact count printed at runtime)

Usage
-----
    # Single session
    feats = extract_video_features("300_OpenFace2.1.0_Pose_gaze_AUs.csv", session_id=300)

    # Full dataset loop (all sessions from labels CSV)
    df = extract_all_sessions(base_url, label_df, save_path="video_features.csv")
"""

import io
import os
import requests
import tarfile
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────

BASE_URL = "https://dcapswoz.ict.usc.edu/wwwedaic/data/"

# OpenFace tracking quality threshold — discard frames below this
CONFIDENCE_THRESHOLD = 0.8

# ── Action Unit column names (continuous intensity, 0.0–5.0) ─────────────────
AU_INTENSITY_COLS = [
    "AU01_r",  # inner brow raise
    "AU02_r",  # outer brow raise
    "AU04_r",  # brow lowerer          → negative affect / concentration
    "AU05_r",  # upper lid raiser
    "AU06_r",  # cheek raiser          → Duchenne smile component
    "AU07_r",  # lid tightener
    "AU09_r",  # nose wrinkler
    "AU10_r",  # upper lip raiser
    "AU12_r",  # lip corner puller     → Duchenne smile component
    "AU14_r",  # dimpler
    "AU15_r",  # lip corner depressor  → sadness marker
    "AU17_r",  # chin raiser           → distress marker
    "AU20_r",  # lip stretcher
    "AU23_r",  # lip tightener
    "AU25_r",  # lips part
    "AU26_r",  # jaw drop
    "AU45_r",  # blink
]

# ── Action Unit column names (binary presence, 0/1) ──────────────────────────
AU_BINARY_COLS = [col.replace("_r", "_c") for col in AU_INTENSITY_COLS]

# ── Head pose columns ─────────────────────────────────────────────────────────
POSE_COLS = ["pose_Rx", "pose_Ry", "pose_Rz"]   # pitch, yaw, roll

# ── Gaze columns ──────────────────────────────────────────────────────────────
GAZE_COLS = ["gaze_angle_x", "gaze_angle_y"]

# ── Clinically meaningful AU composites ──────────────────────────────────────
# Duchenne smile = AU06 + AU12 both active at same time
SMILE_AUS       = ["AU06_r", "AU12_r"]
# Negative affect = AU04 (brow lower) + AU15 (lip corner depress) + AU17 (chin raise)
NEG_AFFECT_AUS  = ["AU04_r", "AU15_r", "AU17_r"]
# Psychomotor agitation proxy = AU01+AU02 (brow movement) + AU25 (jaw)
AGITATION_AUS   = ["AU01_r", "AU02_r", "AU25_r"]


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: LOAD & CLEAN ONE SESSION'S OPENFACE CSV
# ─────────────────────────────────────────────────────────────────────────────

def load_openface_csv(filepath_or_buffer) -> pd.DataFrame:
    """
    Load and clean one OpenFace CSV file.

    Cleaning steps:
      - Strip whitespace from column names (OpenFace adds spaces)
      - Filter frames where confidence < CONFIDENCE_THRESHOLD
      - Filter frames where success == 0 (tracking failed)
      - Reset index after filtering

    Returns cleaned DataFrame. Raises ValueError if too few valid frames remain.
    """
    df = pd.read_csv(filepath_or_buffer)

    # Strip leading/trailing spaces from column names — OpenFace quirk
    df.columns = df.columns.str.strip()

    original_len = len(df)

    # Keep only high-confidence, successful tracking frames
    if "confidence" in df.columns:
        df = df[df["confidence"] >= CONFIDENCE_THRESHOLD]
    if "success" in df.columns:
        df = df[df["success"] == 1]

    df = df.reset_index(drop=True)
    valid_len = len(df)

    print(f"    Frames: {original_len} total → {valid_len} valid "
          f"({100*valid_len/original_len:.1f}% kept)")

    if valid_len < 100:
        raise ValueError(
            f"Only {valid_len} valid frames — too few for reliable features. "
            f"Session will be skipped."
        )

    return df


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: COLUMN PRESENCE CHECK
# ─────────────────────────────────────────────────────────────────────────────

def verify_columns(df: pd.DataFrame) -> dict:
    """
    Check which expected columns are present.
    Returns a dict of flags so downstream code can skip missing groups gracefully.
    """
    present = set(df.columns)

    au_intensity_present = [c for c in AU_INTENSITY_COLS if c in present]
    au_binary_present    = [c for c in AU_BINARY_COLS    if c in present]
    pose_present         = [c for c in POSE_COLS         if c in present]
    gaze_present         = [c for c in GAZE_COLS         if c in present]

    status = {
        "au_intensity" : au_intensity_present,
        "au_binary"    : au_binary_present,
        "pose"         : pose_present,
        "gaze"         : gaze_present,
    }

    for group, cols in status.items():
        expected = {
            "au_intensity": AU_INTENSITY_COLS,
            "au_binary"   : AU_BINARY_COLS,
            "pose"        : POSE_COLS,
            "gaze"        : GAZE_COLS,
        }[group]
        missing = set(expected) - set(cols)
        if missing:
            print(f"    WARNING: {len(missing)} columns missing from {group}: {missing}")

    return status



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: STATISTICAL AGGREGATION HELPER
# ─────────────────────────────────────────────────────────────────────────────

def aggregate_stats(df: pd.DataFrame, cols: list, prefix: str) -> dict:
    """
    For each column in `cols`, compute: mean, std, min, max, range, median, p25, p75.
    Returns flat dict with keys like  "{prefix}_{col}_mean" etc.

    Why these stats?
    - mean    : average expression level across the interview
    - std     : variability / expressiveness (depression → reduced std)
    - min/max : range of expression
    - range   : max-min, direct measure of expressiveness
    - median  : robust central tendency (less affected by outliers)
    - p25/p75 : interquartile range for robust variability
    """
    features = {}
    cols_present = [c for c in cols if c in df.columns]

    for col in cols_present:
        vals = df[col].dropna().values
        if len(vals) == 0:
            for stat in ["mean","std","min","max","range","median","p25","p75"]:
                features[f"{prefix}_{col}_{stat}"] = np.nan
            continue

        features[f"{prefix}_{col}_mean"]   = float(np.mean(vals))
        features[f"{prefix}_{col}_std"]    = float(np.std(vals))
        features[f"{prefix}_{col}_min"]    = float(np.min(vals))
        features[f"{prefix}_{col}_max"]    = float(np.max(vals))
        features[f"{prefix}_{col}_range"]  = float(np.max(vals) - np.min(vals))
        features[f"{prefix}_{col}_median"] = float(np.median(vals))
        features[f"{prefix}_{col}_p25"]    = float(np.percentile(vals, 25))
        features[f"{prefix}_{col}_p75"]    = float(np.percentile(vals, 75))

    return features


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: AU-SPECIFIC FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def extract_au_features(df: pd.DataFrame, au_intensity: list, au_binary: list) -> dict:
    """
    Extract Action Unit features:
      A) Statistical aggregation of all 17 AU intensities
      B) Binary presence rate (% of frames each AU is active)
      C) Clinical composites:
           - Duchenne smile rate  : % frames where AU06 AND AU12 both > 1.0
           - Negative affect score: mean of AU04 + AU15 + AU17 composite
           - Smile intensity mean : mean of AU06+AU12 when smile is active
      D) AU dynamics:
           - Activation rate      : transitions 0→1 per minute (per AU binary)
           - Mean bout duration   : average consecutive-active-frame run length
    """
    features = {}

    # ── A. Statistical aggregation ────────────────────────────────────────────
    features.update(aggregate_stats(df, au_intensity, prefix="au_int"))

    # ── B. Binary presence rate ───────────────────────────────────────────────
    for col in au_binary:
        if col in df.columns:
            features[f"au_bin_{col}_rate"] = float(df[col].mean())

    # ── C. Clinical composites ────────────────────────────────────────────────

    # Duchenne smile: AU06r > 1.0 AND AU12r > 1.0 simultaneously
    if "AU06_r" in df.columns and "AU12_r" in df.columns:
        smile_mask = (df["AU06_r"] > 1.0) & (df["AU12_r"] > 1.0)
        features["au_comp_duchenne_smile_rate"] = float(smile_mask.mean())

        # Mean smile intensity when smiling
        if smile_mask.sum() > 0:
            smile_intensity = (df.loc[smile_mask, "AU06_r"] +
                               df.loc[smile_mask, "AU12_r"]) / 2
            features["au_comp_smile_intensity_mean"] = float(smile_intensity.mean())
        else:
            features["au_comp_smile_intensity_mean"] = 0.0

    # Negative affect composite: mean(AU04, AU15, AU17) per frame → session mean
    neg_cols = [c for c in NEG_AFFECT_AUS if c in df.columns]
    if neg_cols:
        neg_composite = df[neg_cols].mean(axis=1)
        features["au_comp_neg_affect_mean"] = float(neg_composite.mean())
        features["au_comp_neg_affect_std"]  = float(neg_composite.std())

    # Agitation composite: AU01 + AU02 + AU25
    agit_cols = [c for c in AGITATION_AUS if c in df.columns]
    if agit_cols:
        agit_composite = df[agit_cols].mean(axis=1)
        features["au_comp_agitation_mean"] = float(agit_composite.mean())
        features["au_comp_agitation_std"]  = float(agit_composite.std())

    # ── D. AU dynamics (using binary columns) ─────────────────────────────────
    # Compute transitions and bout durations for key AUs only (computational cost)
    key_aus_binary = ["AU04_c", "AU06_c", "AU12_c", "AU15_c", "AU17_c"]

    # Estimate fps from timestamp column if available
    fps = 30.0  # default OpenFace fps
    if "timestamp" in df.columns and len(df) > 1:
        diffs = df["timestamp"].diff().dropna()
        median_diff = diffs.median()
        if median_diff > 0:
            fps = round(1.0 / median_diff)

    for col in key_aus_binary:
        if col not in df.columns:
            continue
        binary_seq = df[col].fillna(0).astype(int).values

        # Activation rate: number of 0→1 transitions per minute
        transitions = np.sum(np.diff(binary_seq) == 1)
        duration_min = len(binary_seq) / (fps * 60)
        features[f"au_dyn_{col}_activation_rate"] = (
            float(transitions / duration_min) if duration_min > 0 else 0.0
        )

        # Mean bout duration: average length of consecutive active runs
        bouts = []
        count = 0
        for val in binary_seq:
            if val == 1:
                count += 1
            else:
                if count > 0:
                    bouts.append(count)
                    count = 0
        if count > 0:
            bouts.append(count)

        features[f"au_dyn_{col}_mean_bout_frames"] = (
            float(np.mean(bouts)) if bouts else 0.0
        )
        features[f"au_dyn_{col}_num_bouts"] = float(len(bouts))

    return features



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: HEAD POSE FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def extract_pose_features(df: pd.DataFrame, pose_cols: list) -> dict:
    """
    Extract head pose features:
      A) Statistical aggregation of Rx, Ry, Rz
      B) Head movement velocity: frame-to-frame angular change
      C) Stillness: % frames with near-zero movement (psychomotor retardation proxy)

    Clinical relevance:
      - Reduced head movement range → psychomotor retardation (PHQ_8Moving)
      - Downward head tilt (negative Rx) → avoidance / submissive posture
      - High movement variance → psychomotor agitation
    """
    features = {}

    # ── A. Statistics ─────────────────────────────────────────────────────────
    features.update(aggregate_stats(df, pose_cols, prefix="pose"))

    # ── B. Head movement velocity ─────────────────────────────────────────────
    if all(c in df.columns for c in ["pose_Rx", "pose_Ry", "pose_Rz"]):
        pose_vals = df[["pose_Rx", "pose_Ry", "pose_Rz"]].values

        # Euclidean distance between consecutive frames
        deltas = np.diff(pose_vals, axis=0)
        velocity = np.sqrt((deltas ** 2).sum(axis=1))  # rad/frame

        features["pose_velocity_mean"]   = float(np.mean(velocity))
        features["pose_velocity_std"]    = float(np.std(velocity))
        features["pose_velocity_max"]    = float(np.max(velocity))
        features["pose_velocity_p90"]    = float(np.percentile(velocity, 90))

        # Stillness: % frames with velocity < 0.01 radians
        # (proxy for psychomotor retardation — very still head)
        features["pose_stillness_rate"] = float(np.mean(velocity < 0.01))

        # Total path length (cumulative head movement over interview)
        features["pose_total_path_rad"] = float(np.sum(velocity))

    # ── C. Mean gaze direction (downward tilt proxy) ──────────────────────────
    # Negative pose_Rx = head pitched downward = avoidance posture
    if "pose_Rx" in df.columns:
        features["pose_downward_tilt_rate"] = float((df["pose_Rx"] < -0.1).mean())

    return features



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: GAZE FEATURES
# ─────────────────────────────────────────────────────────────────────────────

def extract_gaze_features(df: pd.DataFrame, gaze_cols: list) -> dict:
    """
    Extract gaze features:
      A) Statistical aggregation of gaze_angle_x, gaze_angle_y
      B) Gaze instability: std of gaze over short windows (attention proxy)
      C) Aversion rate: % frames where gaze deviates far from center (>0.3 rad)

    Clinical relevance:
      - Reduced gaze variability → reduced engagement / anhedonia
      - High gaze instability → concentration difficulty (PHQ_8Concentrating)
      - Gaze aversion → social withdrawal
    """
    features = {}

    # ── A. Statistics ─────────────────────────────────────────────────────────
    features.update(aggregate_stats(df, gaze_cols, prefix="gaze"))

    if not all(c in df.columns for c in ["gaze_angle_x", "gaze_angle_y"]):
        return features

    gx = df["gaze_angle_x"].values
    gy = df["gaze_angle_y"].values

    # ── B. Gaze instability (rolling window std) ──────────────────────────────
    # Use 90-frame (~3 second) window to capture short-term instability
    window = 90
    if len(gx) >= window:
        gaze_combined = np.sqrt(gx**2 + gy**2)  # gaze deviation magnitude
        rolling_std = pd.Series(gaze_combined).rolling(window).std().dropna().values
        features["gaze_instability_mean"] = float(np.mean(rolling_std))
        features["gaze_instability_max"]  = float(np.max(rolling_std))
    else:
        features["gaze_instability_mean"] = float(np.std(np.sqrt(gx**2 + gy**2)))
        features["gaze_instability_max"]  = np.nan

    # ── C. Gaze aversion rate ─────────────────────────────────────────────────
    # Gaze deviation from center > 0.3 rad in x OR y = looking away
    aversion_mask = (np.abs(gx) > 0.3) | (np.abs(gy) > 0.3)
    features["gaze_aversion_rate"] = float(np.mean(aversion_mask))

    # Downward gaze: gy > 0.2 rad = looking down (submissive/avoidant)
    features["gaze_downward_rate"] = float(np.mean(gy > 0.2))

    # Gaze velocity (frame-to-frame change)
    gaze_vel = np.sqrt(np.diff(gx)**2 + np.diff(gy)**2)
    features["gaze_velocity_mean"] = float(np.mean(gaze_vel))
    features["gaze_velocity_std"]  = float(np.std(gaze_vel))

    return features


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: FULL FEATURE EXTRACTION FOR ONE SESSION
# ─────────────────────────────────────────────────────────────────────────────

def extract_video_features(filepath_or_buffer, session_id: int) -> Optional[dict]:
    """
    Master function: load + clean + extract all video features for one session.

    Parameters
    ----------
    filepath_or_buffer : str, Path, or file-like object
        Path to XXX_OpenFace2.1.0_Pose_gaze_AUs.csv  OR  a BytesIO buffer
    session_id : int
        Participant ID (e.g. 300)

    Returns
    -------
    dict of {feature_name: float}  with session_id prepended,
    or None if the session fails quality checks.
    """
    print(f"\n[Session {session_id}] Extracting video features...")

    try:
        # ── Load & clean ──────────────────────────────────────────────────────
        df = load_openface_csv(filepath_or_buffer)

        # ── Verify columns ────────────────────────────────────────────────────
        status = verify_columns(df)

        # ── Extract all feature groups ────────────────────────────────────────
        all_features = {"Participant_ID": session_id}

        # Action Units
        au_feats = extract_au_features(
            df,
            au_intensity=status["au_intensity"],
            au_binary=status["au_binary"]
        )
        all_features.update(au_feats)

        # Head pose
        pose_feats = extract_pose_features(df, status["pose"])
        all_features.update(pose_feats)

        # Gaze
        gaze_feats = extract_gaze_features(df, status["gaze"])
        all_features.update(gaze_feats)

        # Total valid frames (useful for downstream quality weighting)
        all_features["video_valid_frames"] = len(df)
        all_features["video_duration_sec"] = (
            float(df["timestamp"].max()) if "timestamp" in df.columns else np.nan
        )

        print(f"    → {len(all_features) - 3} features extracted successfully")
        return all_features

    except Exception as e:
        print(f"    ERROR: {e}")
        return None


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: FULL DATASET LOOP — STREAM + EXTRACT ALL SESSIONS
# ─────────────────────────────────────────────────────────────────────────────

def stream_openface_from_tar(session_id: int) -> Optional[io.BytesIO]:
    """
    Stream a session tar.gz from the E-DAIC server and extract
    only the OpenFace CSV — without saving the full tar to disk.

    Returns BytesIO buffer of the CSV, or None on failure.
    """
    url = f"{BASE_URL}{session_id}_P.tar.gz"
    target_file = f"{session_id}_P/features/{session_id}_OpenFace2.1.0_Pose_gaze_AUs.csv"

    print(f"  Streaming {url}...", end=" ")
    try:
        response = requests.get(url, stream=True, timeout=300)
        if response.status_code != 200:
            print(f"HTTP {response.status_code}")
            return None

        chunks = []
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            chunks.append(chunk)
        buffer = io.BytesIO(b"".join(chunks))
        print(f"downloaded ({sum(len(c) for c in chunks)/1e6:.0f} MB)")

        with tarfile.open(fileobj=buffer) as tar:
            try:
                member = tar.getmember(target_file)
                f = tar.extractfile(member)
                return io.BytesIO(f.read())
            except KeyError:
                print(f"    File not found in tar: {target_file}")
                return None

    except Exception as e:
        print(f"    Stream error: {e}")
        return None


def extract_all_sessions(
    label_df: pd.DataFrame,
    save_path: str = "video_features.csv",
    checkpoint_every: int = 10
) -> pd.DataFrame:
    """
    Loop over all sessions in label_df, stream each tar, extract video features.

    Parameters
    ----------
    label_df      : DataFrame with at least a 'Participant_ID' column
    save_path     : where to save the accumulated feature CSV
    checkpoint_every : save intermediate results every N sessions

    Returns
    -------
    DataFrame of shape [n_sessions x n_features]
    """
    session_ids = label_df["Participant_ID"].tolist()
    results = []
    failed = []

    print(f"Processing {len(session_ids)} sessions...\n")

    for i, sid in enumerate(session_ids):
        # Stream and extract
        csv_buffer = stream_openface_from_tar(int(sid))
        if csv_buffer is None:
            failed.append(sid)
            continue

        feats = extract_video_features(csv_buffer, session_id=int(sid))
        if feats is None:
            failed.append(sid)
            continue

        results.append(feats)

        # Checkpoint save
        if (i + 1) % checkpoint_every == 0:
            checkpoint_df = pd.DataFrame(results)
            checkpoint_df.to_csv(save_path, index=False)
            print(f"\n  [Checkpoint] Saved {len(results)} sessions to {save_path}")

    # Final save
    feature_df = pd.DataFrame(results)
    feature_df.to_csv(save_path, index=False)

    print(f"\n{'='*60}")
    print(f"Done. {len(results)} sessions extracted, {len(failed)} failed.")
    if failed:
        print(f"Failed sessions: {failed}")
    print(f"Feature matrix shape: {feature_df.shape}")
    print(f"Saved to: {save_path}")

    return feature_df


In [ ]:













# ─────────────────────────────────────────────────────────────────────────────
# STEP 9: POST-PROCESSING — NORMALISATION & MERGE WITH LABELS
# ─────────────────────────────────────────────────────────────────────────────

def postprocess_features(
    feature_df: pd.DataFrame,
    label_df: pd.DataFrame,
    nan_strategy: str = "median"
) -> pd.DataFrame:
    """
    Post-process extracted features:
      1. Merge with labels on Participant_ID
      2. Handle NaN values (median imputation by default)
      3. Z-score normalise all feature columns (fit on train, apply to dev/test)

    Parameters
    ----------
    feature_df   : output of extract_all_sessions()
    label_df     : merged labels DataFrame with 'split' column
    nan_strategy : 'median' | 'mean' | 'zero'

    Returns
    -------
    Merged and normalised DataFrame
    """
    # ── Merge ─────────────────────────────────────────────────────────────────
    merged = label_df.merge(feature_df, on="Participant_ID", how="inner")
    print(f"Merged: {len(merged)} sessions with both features and labels")

    # Identify feature columns (exclude metadata)
    meta_cols = ["Participant_ID", "Gender", "PHQ_Binary", "PHQ_Score",
                 "PCL-C (PTSD)", "PTSD Severity", "split",
                 "PHQ_8NoInterest", "PHQ_8Depressed", "PHQ_8Sleep",
                 "PHQ_8Tired", "PHQ_8Appetite", "PHQ_8Failure",
                 "PHQ_8Concentrating", "PHQ_8Moving", "PHQ_8Total"]
    feature_cols = [c for c in merged.columns if c not in meta_cols]

    print(f"Feature columns: {len(feature_cols)}")
    print(f"NaN count before imputation: {merged[feature_cols].isna().sum().sum()}")

    # ── NaN imputation ────────────────────────────────────────────────────────
    # Compute fill values from TRAIN split only to avoid data leakage
    train_mask = merged["split"] == "train"
    for col in feature_cols:
        if merged[col].isna().any():
            if nan_strategy == "median":
                fill_val = merged.loc[train_mask, col].median()
            elif nan_strategy == "mean":
                fill_val = merged.loc[train_mask, col].mean()
            else:
                fill_val = 0.0
            merged[col] = merged[col].fillna(fill_val)

    # ── Z-score normalisation ─────────────────────────────────────────────────
    # Fit mean/std on train only
    train_mean = merged.loc[train_mask, feature_cols].mean()
    train_std  = merged.loc[train_mask, feature_cols].std().replace(0, 1)

    merged[feature_cols] = (merged[feature_cols] - train_mean) / train_std

    print(f"NaN count after imputation: {merged[feature_cols].isna().sum().sum()}")
    print(f"Final feature matrix: {merged[feature_cols].shape}")

    return merged







In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 10: FEATURE SUMMARY & CLINICAL INTERPRETATION
# ─────────────────────────────────────────────────────────────────────────────

def summarise_features(feature_df: pd.DataFrame, label_df: pd.DataFrame):
    """
    Print a clinical summary: mean feature values for depressed vs non-depressed.
    Useful for sanity checking — depressed group should show:
      - Lower duchenne_smile_rate
      - Higher neg_affect_mean
      - Lower pose_velocity_mean (less head movement)
      - Higher gaze_aversion_rate
    """
    merged = label_df.merge(feature_df, on="Participant_ID", how="inner")
    dep   = merged[merged["PHQ_Binary"] == 1]
    nodep = merged[merged["PHQ_Binary"] == 0]

    key_features = [
        "au_comp_duchenne_smile_rate",
        "au_comp_neg_affect_mean",
        "au_comp_agitation_mean",
        "pose_velocity_mean",
        "pose_stillness_rate",
        "gaze_aversion_rate",
        "gaze_instability_mean",
    ]

    print(f"\n{'Feature':<40} {'Depressed':>12} {'Not Depressed':>14} {'Direction':>12}")
    print("-" * 80)
    for feat in key_features:
        if feat not in merged.columns:
            continue
        d_mean  = dep[feat].mean()
        nd_mean = nodep[feat].mean()
        direction = "↑ depressed" if d_mean > nd_mean else "↓ depressed"
        print(f"{feat:<40} {d_mean:>12.4f} {nd_mean:>14.4f} {direction:>12}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN — EXAMPLE RUN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys

    # ── Load labels ───────────────────────────────────────────────────────────
    print("Loading labels...")
    labels_base = "https://dcapswoz.ict.usc.edu/wwwedaic/labels/"

    def load_label_csv(name):
        r = requests.get(labels_base + name)
        return pd.read_csv(io.StringIO(r.text))

    train_df   = load_label_csv("train_split.csv");   train_df["split"]  = "train"
    dev_df     = load_label_csv("dev_split.csv");     dev_df["split"]    = "dev"
    test_df    = load_label_csv("test_split.csv");    test_df["split"]   = "test"
    detailed   = load_label_csv("Detailed_PHQ8_Labels.csv")

    all_labels = pd.concat([train_df, dev_df, test_df], ignore_index=True)
    all_labels = all_labels.merge(detailed, on="Participant_ID", how="left")

    print(f"Total sessions: {len(all_labels)}")

    # ── Option A: Test on a single session first ──────────────────────────────
    if "--single" in sys.argv:
        test_session = 300
        csv_buf = stream_openface_from_tar(test_session)
        if csv_buf:
            feats = extract_video_features(csv_buf, session_id=test_session)
            if feats:
                print(f"\nExtracted {len(feats)} features for session {test_session}")
                print("\nSample features:")
                for k, v in list(feats.items())[:20]:
                    print(f"  {k:<50} {v:.4f}")

    # ── Option B: Full dataset extraction ─────────────────────────────────────
    else:
        save_path = "edaic_video_features.csv"
        feature_df = extract_all_sessions(
            label_df=all_labels,
            save_path=save_path,
            checkpoint_every=10
        )

        # Post-process
        processed = postprocess_features(feature_df, all_labels)
        processed.to_csv("edaic_video_features_processed.csv", index=False)

        # Sanity check
        summarise_features(feature_df, all_labels)